# **animove**

### Temporal Patterns of Bobcat (_Lynx rufus_) Movement: Diel Activity, Activity Budgets, and Seasonal Variation Near Black Rock Forest, New York

**By**: Zhean Robby Ganituen, Jaztin Jacob Jimenez, Reece Benedict Orense, Ashley Paulianna Reyes, and Matthew Fraser Sim

---

## **introduction**

**Dataset**: LaPoint, S. 2026. Data from: Study “Carnivore movements near Black Rock Forest New York” [part]. Movebank Data Repository.

**Paper**: Oliver, R.Y. et al. 2026. Interacting effects of human presence and landscape modification on birds and mammals. Science. 392, 6800 (May 2026), 879–884. https://doi.org/10.1126/science.adq3396.

`<TODO :: add dataset description here>`

`<TODO :: add plan with the dataset>`

## **definition of terms**

We define words important to this study below:

- **Activity budget**: Is the proportion at which an animal spends their time moving versus resting.
- **Crepsucular**: Are animals that are primarily active during the twilight hours; not to be confused with _nocturnal_ animals which are active during pitch-black hours.
- **Diel Movement**: Is the synchronized, daily movement of animals moving around their habitat, occuring over the 24-hour cycle in response to light, temperature, and/or food availability.

## **research questions**

The goal of this project is to answer the general question:

> **How is the movement activity of bobcats (_Lynx rufus_) structured in time, across the daily cycle and across seasons, near Black Rock Forest, New York?**

To answer this general question, we have constructed the following research questions:

1. Is there a significant difference in the movement rate of _Lynx rufus_ during the crepuscular periods compared with midday and the middle of the night?
2. What proportion of time do _Lynx rufus_ spend in active travel versus resting, their activity budget, and how much does this vary among individuals?
3. Among the individuals whose records span multiple seasons, does the diel movement pattern change across seasons?

## **0 setup**

Let's first import the Python libraries needed for the project and define some filenames which will be used throughout the dataset.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

# Directories
DIR_DATA = "../data/"
DIR_RAW_DATA = DIR_DATA + "raw/"
DIR_PROC_DATA = DIR_DATA + "processed/"

# Output Filenames
OUT_JOINED_DATASET = DIR_RAW_DATA + "joined.csv"

# DataFrames global variables
MAIN_DF = ()
REF_DF = ()
JOIN_DF = ()
PPROC_DF = ()

### **0.1 dataset loading**

Now, let's load the two raw datasets as a pandas `DataFrame`.

In [2]:
MAIN_DF = pd.read_csv(DIR_RAW_DATA + "main.csv")
REF_DF = pd.read_csv(DIR_RAW_DATA + "reference.csv")

/tmp/ipykernel_36110/795800667.py:1: DtypeWarning: Columns (0: manually-marked-outlier) have mixed types. Specify dtype option on import or set low_memory=False.
  MAIN_DF = pd.read_csv(DIR_RAW_DATA + "main.csv")


### **0.2 joining**

Before doing any cleaning, let's first join the two. 

To get an idea of what the dataset looks like. Let's first print the columns and information from each dataframe. From this, we can come up with a methodology for joining the two.

In [3]:
print("\t========== Main DataFrame ========== ")
print(MAIN_DF.info())

	========== Main DataFrame ========== 
<class 'pandas.DataFrame'>
RangeIndex: 131930 entries, 0 to 131929
Data columns (total 26 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   event-id                           131930 non-null  int64  
 1   visible                            131930 non-null  bool   
 2   timestamp                          131930 non-null  str    
 3   location-long                      126483 non-null  float64
 4   location-lat                       126483 non-null  float64
 5   data-decoding-software             131930 non-null  int64  
 6   eobs:battery-voltage               131930 non-null  int64  
 7   eobs:fix-battery-voltage           131930 non-null  int64  
 8   eobs:horizontal-accuracy-estimate  126483 non-null  float64
 9   eobs:key-bin-checksum              131930 non-null  int64  
 10  eobs:speed-accuracy-estimate       126483 non-null  float64
 11  eobs:start-

In [4]:
print("\t========== Reference DataFrame ========== ")
print(REF_DF.info())

	========== Reference DataFrame ========== 
<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tag-id                 9 non-null      int64  
 1   animal-id              9 non-null      str    
 2   animal-taxon           9 non-null      str    
 3   deploy-on-date         9 non-null      str    
 4   deploy-off-date        8 non-null      str    
 5   animal-life-stage      9 non-null      str    
 6   animal-mass            9 non-null      float64
 7   animal-offspring       1 non-null      str    
 8   animal-parents         2 non-null      str    
 9   animal-sex             9 non-null      str    
 10  animal-siblings        2 non-null      str    
 11  attachment-body-part   9 non-null      str    
 12  attachment-type        9 non-null      str    
 13  deploy-on-latitude     1 non-null      float64
 14  deploy-on-longitude    1 non-

From here, the column `individual-local-identifier` (`ili`) from `MAIN_DF` and `animal-id` from `REF_DF` seem to be related to each other. Let's verify this claim by checking that the set of possible values for each column is the same.

In [5]:
unique_ili_id = set(MAIN_DF["individual-local-identifier"].unique())
unique_animal_id = set(REF_DF["animal-id"].unique())

print("\t========== Is ili and animal-id a potential join key? ==========")
if unique_ili_id == unique_animal_id:
    print("YES")
    print(unique_ili_id)
else:
    print("NO")

	========== Is ili and animal-id a potential join key? ==========
YES
{'PEPE_0001', 'LYRU_0001', 'PEPE_0002', 'LYRU_0004', 'LYRU_0005', 'LYRU_0002', 'LYRU_0003', 'LYRU_0006', 'LYRU_0008'}


Now that we know that `ili` and `animal-id` is the join key. We will now join `MAIN_DF` and `REF_DF` to construct `JOIN_DF`. But let's first rename `ili` to `animal-id`. Then, we'll call `pd.merge` on the join key (which is now `animal-id`).

In [6]:
MAIN_DF = MAIN_DF.rename(columns={"individual-local-identifier": "animal-id"})
JOIN_DF = pd.merge(MAIN_DF, REF_DF, on="animal-id")

Now let's see the columns and information of the dataset after performing the join. Let's ensure that information of the two datasets were kept after merging by performing some tests.

In [7]:
print("\t========== Joined DataFrame ========== ")
print("(Running tests...)")
print(
    "=== The total number of columns must be len(MAIN_DF.columns) + len(REF_DF.columns) - 1"
)
assert len(MAIN_DF.columns) + len(REF_DF.columns) - 1 == len(JOIN_DF.columns)

print("=== The total number of entries must be the max(len(MAIN_DF), len(REF_DF))")
assert max(len(MAIN_DF), len(REF_DF)) == len(JOIN_DF)

print()
print("(Saving joined dataset...)")
JOIN_DF.to_csv(OUT_JOINED_DATASET, index=False)

print()
print("=== JOIN_DF info")
print(JOIN_DF.info())

	========== Joined DataFrame ========== 
(Running tests...)
=== The total number of columns must be len(MAIN_DF.columns) + len(REF_DF.columns) - 1
=== The total number of entries must be the max(len(MAIN_DF), len(REF_DF))

(Saving joined dataset...)

=== JOIN_DF info
<class 'pandas.DataFrame'>
RangeIndex: 131930 entries, 0 to 131929
Data columns (total 49 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   event-id                           131930 non-null  int64  
 1   visible                            131930 non-null  bool   
 2   timestamp                          131930 non-null  str    
 3   location-long                      126483 non-null  float64
 4   location-lat                       126483 non-null  float64
 5   data-decoding-software             131930 non-null  int64  
 6   eobs:battery-voltage               131930 non-null  int64  
 7   eobs:fix-battery-voltage           131930 

## **1 Dataset Preprocessing**

Now that we have the joined dataset, we can proceed with the cleaning phase.


Three source files are important for this section:
- `janitor` defines functions and a shared interface for **cleaning**.
- `transformer` defines functions and a shared interface for **transforming** data into more useful formats.
- `reducer` defines variables and functions for **dimensionality reduction** (removing unnecessary columns).

In [8]:
import janitor
import transformer
import reducer

preprocess_step_df = JOIN_DF

<p style="color: red;">ADD EXPLANATION.</p>

In [9]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_fishers
)
report.out()

	========== Janitor ========== 
		Rows before		: 131930
		Rows remaining		: 116256
		Rows removed		: 15674
		Examples (max 3)	:                                                                               112677                                             112678                                             112679
event-id                                                                 26403030653                                        26403030670                                        26403030672
visible                                                                         True                                               True                                               True
timestamp                                                    2023-03-21 16:00:53.000                            2023-03-21 17:01:11.000                            2023-03-21 18:00:23.000
location-long                                                             -74.048125                                        

<p style="color: red;">ADD EXPLANATION.</p>

In [10]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_bad_gps
)
report.out()

	========== Janitor ========== 
		Rows before		: 116256
		Rows remaining		: 112312
		Rows removed		: 3944
		Examples (max 3)	:                                                                                  25                                                 26                                                 134
event-id                                                                 14056446662                                        14056446663                                        14056446771
visible                                                                         True                                               True                                               True
timestamp                                                    2020-02-12 22:40:15.000                            2020-02-12 22:40:16.000                            2020-02-13 13:36:08.000
location-long                                                                    NaN                                         

<p style="color: red;">ADD EXPLANATION.</p>

In [11]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_with_markers
)
report.out()

	========== Janitor ========== 
		Rows before		: 112312
		Rows remaining		: 112307
		Rows removed		: 5
		Examples (max 3)	:                                                                                36976                                              36977                                              36978
event-id                                                                 18681223130                                        18681223131                                        18681223132
visible                                                                        False                                              False                                              False
timestamp                                                    2021-04-28 16:38:21.000                            2021-04-28 16:40:10.000                            2021-04-28 16:42:10.000
location-long                                                             -74.024605                                         -74

<p style="color: red;">ADD EXPLANATION.</p>

In [12]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_bad_fix
)
report.out()

	========== Janitor ========== 
		Rows before		: 112307
		Rows remaining		: 112232
		Rows removed		: 75
		Examples (max 3)	:                                                                                84338                                              84925                                              84940
event-id                                                                 47041856868                                        47041856869                                        47041856870
visible                                                                         True                                               True                                               True
timestamp                                                    2023-04-27 13:01:20.000                            2023-04-30 21:01:20.000                            2023-05-01 04:29:19.000
location-long                                                             -74.092102                                         -7

<p style="color: red;">ADD EXPLANATION.</p>

In [13]:
preprocess_step_df, report = transformer.transform_runner(
    df=preprocess_step_df, func=transformer.add_utc_time, source_cols=["timestamp"]
)
report.out()

	========== Transformer ========== 
		Added columns	: ['timestamp-utc']
		Before (max 3)	:
                                 0                        1                        2
event-id               14056446637              14056446638              14056446639
timestamp  2020-02-12 16:02:53.000  2020-02-12 16:02:54.000  2020-02-12 16:02:55.000
		After  (max 3)	:
                                       0                          1                          2
event-id                     14056446637                14056446638                14056446639
timestamp        2020-02-12 16:02:53.000    2020-02-12 16:02:54.000    2020-02-12 16:02:55.000
timestamp-utc  2020-02-12 16:02:53+00:00  2020-02-12 16:02:54+00:00  2020-02-12 16:02:55+00:00


<p style="color: red;">ADD EXPLANATION.</p>

In [14]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_outside_deploy
)
report.out()

	========== Janitor ========== 
		Rows before		: 112232
		Rows remaining		: 112232
		Rows removed		: 0
		Examples (max 3)	: Empty DataFrame
Columns: []
Index: [event-id, visible, timestamp, location-long, location-lat, data-decoding-software, eobs:battery-voltage, eobs:fix-battery-voltage, eobs:horizontal-accuracy-estimate, eobs:key-bin-checksum, eobs:speed-accuracy-estimate, eobs:start-timestamp, eobs:status, eobs:temperature, eobs:type-of-fix, eobs:used-time-to-get-fix, ground-speed, heading, height-above-ellipsoid, import-marked-outlier, manually-marked-outlier, sensor-type, individual-taxon-canonical-name, tag-local-identifier, animal-id, study-name, tag-id, animal-taxon, deploy-on-date, deploy-off-date, animal-life-stage, animal-mass, animal-offspring, animal-parents, animal-sex, animal-siblings, attachment-body-part, attachment-type, deploy-on-latitude, deploy-on-longitude, deploy-on-person, deployment-comments, deployment-id, manipulation-type, tag-beacon-frequency, tag-manufact

<p style="color: red;">ADD EXPLANATION.</p>

In [15]:
preprocess_step_df, report = janitor.clean_runner(
    df=preprocess_step_df, func=janitor.rem_dup_sessions
)
report.out()

	========== Janitor ========== 
		Rows before		: 112232
		Rows remaining		: 94490
		Rows removed		: 17742
		Examples (max 3)	:                                                                                    0                                                  1                                                  3
event-id                                                                 14056446637                                        14056446638                                        14056446640
visible                                                                         True                                               True                                               True
timestamp                                                    2020-02-12 16:02:53.000                            2020-02-12 16:02:54.000                            2020-02-12 17:00:25.000
location-long                                                             -74.059272                                         

<p style="color: red;">ADD EXPLANATION.</p>

In [16]:
preprocess_step_df, report = transformer.transform_runner(
    df=preprocess_step_df,
    func=transformer.add_local_features,
    source_cols=["timestamp-utc"],
)

report.out()

	========== Transformer ========== 
		Added columns	: ['timestamp-local', 'hour-local', 'date-local', 'month-local', 'season']
		Before (max 3)	:
                                       0                          1                          2
event-id                     14078351206                47041858690                19885952950
timestamp-utc  2020-02-27 04:00:25+00:00  2026-03-08 06:50:23+00:00  2021-08-31 08:00:23+00:00
		After  (max 3)	:
                                         0                          1                          2
event-id                       14078351206                47041858690                19885952950
timestamp-utc    2020-02-27 04:00:25+00:00  2026-03-08 06:50:23+00:00  2021-08-31 08:00:23+00:00
timestamp-local  2020-02-26 23:00:25-05:00  2026-03-08 01:50:23-05:00  2021-08-31 04:00:23-04:00
hour-local                              23                          1                          4
date-local                      2020-02-26                 2026-0

<p style="color: red;">ADD EXPLANATION.</p>

In [17]:
preprocess_step_df, report = transformer.transform_runner(
    df=preprocess_step_df,
    func=transformer.add_movement,
    source_cols=["timestamp-utc", "location-lat", "location-long", "ground-speed"],
)

report.out()

	========== Transformer ========== 
		Added columns	: ['dt-seconds', 'step-meters', 'derived-speed-ms']
		Before (max 3)	:
                                   35052                      59139                      62773
event-id                     14056446639                14056446642                14056446645
timestamp-utc  2020-02-12 16:02:55+00:00  2020-02-12 17:00:27+00:00  2020-02-12 18:00:49+00:00
location-lat                   41.380708                   41.38084                  41.380899
location-long                 -74.059251                 -74.059188                 -74.059135
ground-speed                        0.09                       0.04                       0.24
		After  (max 3)	:
                                          0                          1                          2
event-id                        14056446639                14056446642                14056446645
timestamp-utc     2020-02-12 16:02:55+00:00  2020-02-12 17:00:27+00:00  2020-02-12 18:00:49+

As the final step, let's reduce the dimension of the dataset by only keeping columns important for the research questions and some other columns that can help us make sense of the dataset.

These columns are:

- <p style="color: red;">write each column name and explain why it's important.</p>


<p style="color: red;">NOTE. I added a preliminary list already. cross check if it makes sense.</p>


In [22]:
keep_cols = [
    "event-id",
    "animal-id",
    "individual-taxon-canonical-name",
    "timestamp",
    "timestamp-utc",
    "timestamp-local",
    "hour-local",
    "date-local",
    "month-local",
    "season",
    "ground-speed",
    "heading",
    "dt-seconds",
    "step-meters",
    "derived-speed-ms",
    "location-long",
    "location-lat",
    "animal-sex",
    "animal-life-stage",
    "animal-mass",
    "deploy-on-date",
    "deploy-off-date",
    "eobs:start-timestamp",
    "eobs:horizontal-accuracy-estimate",
    "eobs:temperature",
]

In [24]:
preprocess_step_df = reducer.reduce(df=preprocess_step_df, cols=keep_cols)

assert len(preprocess_step_df.columns) == len(keep_cols)

Now that we have performed all cleaning steps, let's store the final result in `PPROC_DF` and check its information.

In [25]:
PPROC_DF = preprocess_step_df

print("\t========== Preprocessed DataFrame ========== ")
print(PPROC_DF.info())

	========== Preprocessed DataFrame ========== 
<class 'pandas.DataFrame'>
RangeIndex: 94490 entries, 0 to 94489
Data columns (total 25 columns):
 #   Column                             Non-Null Count  Dtype                           
---  ------                             --------------  -----                           
 0   event-id                           94490 non-null  int64                           
 1   animal-id                          94490 non-null  str                             
 2   individual-taxon-canonical-name    94490 non-null  str                             
 3   timestamp                          94490 non-null  str                             
 4   timestamp-utc                      94490 non-null  datetime64[us, UTC]             
 5   timestamp-local                    94490 non-null  datetime64[us, America/New_York]
 6   hour-local                         94490 non-null  int32                           
 7   date-local                         94490 non-null

# **n Exploratory Data Analysis**

<??>
What else do we add in this notebook?

The researcj questions?